In [2]:
!pip install undetected-chromedriver

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 13.0 MB/s eta 0:00:00
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47047 sha256=81c4f8104b154605f92ee175e3dad80d515a46c6d775677c2a1f65cb540586ac
  Stored in directory: /root/.cache/pip/wheels/c4/f1/aa/9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected-chromedriver
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [4]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import pandas as pd
import time
import random
import requests
from bs4 import BeautifulSoup
import concurrent.futures

In [7]:
import undetected_chromedriver as uc

options = uc.ChromeOptions()

driver = uc.Chrome(
    options=options,
    browser_executable_path="/usr/bin/google-chrome"
)

FileNotFoundError: 
---------------------
Could not determine browser executable.
---------------------
Make sure your browser is installed in the default location (path).
If you are sure about the browser executable, you can specify it using
the `browser_executable_path='/path/to/browser/executable` parameter.



In [6]:
# 1. Initialize the Stealth Browser
options = uc.ChromeOptions()
driver = uc.Chrome(options=options)

congress_map = {
    "18th Congress (July 2019 - 2022)": "220",
    "17th Congress (July 2016 - 2019)": "53",
    "16th Congress (July 2013 - 2016)": "240",
    "15th Congress (July 2010 - 2013)": "48"
}

base_domain = "https://ldr.senate.gov.ph"

TypeError: Binary Location Must be a String

In [ ]:
# 2. Loop through each Congress
for congress_label, target_id in congress_map.items():
    print(f"\n========================================================")
    print(f"🎬 Activating Protocol for: {congress_label}")
    print(f"========================================================")

    current_page_url = f"{base_domain}/legislative-issuances/senate-bills?field_cn_bill_was_filed_target_id={target_id}&field_bill_date_of_filing_value=All&sort_by=field_bill_date_of_filing_value&sort_order=DESC&items_per_page=50"

    bill_urls_to_scrape = []
    page_number = 1

    # ---------------------------------------------------------
    # PHASE 1: THE SCOUT (Collect URLs)
    # ---------------------------------------------------------
    print("📍 Phase 1: Scouting all bill URLs...")
    while True:
        driver.get(current_page_url)

        while "Just a moment" in driver.title or "Cloudflare" in driver.title:
            print("🚨 Cloudflare intercepted! Waiting 5s...")
            time.sleep(5)

        time.sleep(random.uniform(2.0, 3.5))
        print(f"   Scanning page {page_number}...")

        bill_links = driver.find_elements(By.XPATH, "//h2/a")
        for link in bill_links:
            href = link.get_attribute("href")
            if href and href not in bill_urls_to_scrape:
                bill_urls_to_scrape.append(href)

        try:
            next_button = driver.find_elements(By.XPATH, "//li[contains(@class, 'pager__item--next')]/a")
            if len(next_button) > 0:
                current_page_url = next_button[0].get_attribute("href")
                page_number += 1
            else:
                print(f"   End of pagination reached. Total bills found: {len(bill_urls_to_scrape)}")
                break
        except Exception as e:
            print("   No next button found.")
            break

    # ---------------------------------------------------------
    # THE COOKIE HAND-OFF
    # ---------------------------------------------------------
    print("\n   [Transferring Security Clearance to High-Speed Parser...]")
    session = requests.Session()

    def copy_clearance_to_session():
        session.cookies.clear()
        for cookie in driver.get_cookies():
            session.cookies.set(cookie['name'], cookie['value'])
        user_agent = driver.execute_script("return navigator.userAgent;")
        session.headers.update({"User-Agent": user_agent})

    copy_clearance_to_session()

    # ---------------------------------------------------------
    # PHASE 2: THE DEEP DIVE (Auto-Refresher & Dynamic Queue)
    # ---------------------------------------------------------
    print(f"📍 Phase 2: Turbo-scraping {len(bill_urls_to_scrape)} bills...")
    all_bills_data = []

    # --- AUTO-REFRESH FUNCTION ---
    def refresh_clearance():
        print("\n   [🔄 403 DETECTED: VIP Pass Expired!]")
        print("   [🤖 Waking up Stealth Browser to get a new pass from Cloudflare...]")
        driver.get(base_domain) # Ping the site to trigger a check
        time.sleep(5)
        while "Just a moment" in driver.title or "Cloudflare" in driver.title:
            print("   🚨 Cloudflare intercepted! Waiting for auto-solve...")
            time.sleep(5)
        copy_clearance_to_session()
        print("   [✅ Security Clearance Renewed! Resuming scrape...]")
    # -----------------------------

    def scrape_single_bill(bill_url):
        try:
            time.sleep(random.uniform(0.5, 1.5))
            response = session.get(bill_url, timeout=15)

            # If we get a 403 or Timeout, return a 'BLOCKED' status so the queue knows to retry it
            if response.status_code == 403 or response.status_code == 503:
                return {"status": "BLOCKED", "url": bill_url}
            if response.status_code != 200:
                return {"status": "ERROR", "url": bill_url}

            soup = BeautifulSoup(response.text, 'html.parser')

            # Bulletproof Matcher
            def get_bs4_field(label_name):
                for div in soup.find_all("div", class_=True):
                    class_str = " ".join(div["class"]).lower()
                    div_text = div.get_text(strip=True).lower()
                    if "label" in class_str and label_name.lower() in div_text:
                        item_div = div.find_next_sibling("div", class_=True)
                        if item_div:
                            item_class_str = " ".join(item_div["class"]).lower()
                            if "item" in item_class_str:
                                return " ".join(item_div.get_text(separator=' ', strip=True).split())
                return "N/A"

            bill_no = "N/A"
            headers = soup.find_all("h1", class_="au-header-heading")
            for header in headers:
                text = header.get_text(strip=True)
                if "Senate Bill No." in text:
                    bill_no = text.split(",")[0].strip()
                    break

            data = {
                "Bill_No": bill_no,
                "Congress_Name": congress_label,
                "Bill_URL": bill_url,
                "Short_Title": get_bs4_field("Short Title"),
                "Long_Title": get_bs4_field("Long Title"),
                "Author": get_bs4_field("Author"),
                "Date_Filed": get_bs4_field("Date filed"),
                "Scope": get_bs4_field("Scope"),
                "Subjects": get_bs4_field("Subjects"),
                "Primary_Committee": get_bs4_field("Primary Committee"),
                "Legislative_Status": get_bs4_field("Legislative Status")
            }
            return {"status": "SUCCESS", "data": data}

        except requests.exceptions.Timeout:
            return {"status": "BLOCKED", "url": bill_url} # Treat timeout as a block to retry
        except Exception as e:
            return {"status": "ERROR", "url": bill_url}

    # --- DYNAMIC QUEUE LOGIC ---
    urls_to_process = bill_urls_to_scrape.copy()
    batch_size = 25 # Safe volume

    while len(urls_to_process) > 0:
        # Pull the next batch of URLs from the front of the line
        batch = urls_to_process[:batch_size]
        urls_to_process = urls_to_process[batch_size:] # Remove them from the pending queue

        print(f"\n   [🚀 Scraping next {len(batch)} bills... {len(urls_to_process)} remaining in queue]")

        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            results = list(executor.map(scrape_single_bill, batch))

        blocked_urls = []

        for res in results:
            if res:
                if res["status"] == "SUCCESS":
                    all_bills_data.append(res["data"])
                elif res["status"] == "BLOCKED":
                    blocked_urls.append(res["url"])

        # If any URLs got a 403, we refresh and put them back in line!
        if blocked_urls:
            print(f"   [!] {len(blocked_urls)} bills were blocked in this batch.")
            refresh_clearance()
            # Add the blocked URLs back to the VERY FRONT of the queue
            urls_to_process = blocked_urls + urls_to_process
            time.sleep(5)
        else:
            print(f"   [⏳ Batch perfectly clean. Breathing for 8 seconds...]")
            time.sleep(8)



🎬 Activating Protocol for: 16th Congress (July 2013 - 2016)
📍 Phase 1: Scouting all bill URLs...
   Scanning page 1...
   Scanning page 2...
   Scanning page 3...
   Scanning page 4...
   Scanning page 5...
   Scanning page 6...
   Scanning page 7...
   Scanning page 8...
   Scanning page 9...
   Scanning page 10...
   Scanning page 11...
   Scanning page 12...
   Scanning page 13...
   Scanning page 14...
   Scanning page 15...
   Scanning page 16...
   Scanning page 17...
   Scanning page 18...
   Scanning page 19...
   Scanning page 20...
   Scanning page 21...
   Scanning page 22...
   Scanning page 23...
   Scanning page 24...
   Scanning page 25...
   Scanning page 26...
   Scanning page 27...
   Scanning page 28...
   Scanning page 29...
   Scanning page 30...
   Scanning page 31...
   Scanning page 32...
   Scanning page 33...
   Scanning page 34...
   Scanning page 35...
   Scanning page 36...
   Scanning page 37...
   Scanning page 38...
   Scanning page 39...
   Scanning pa

In [ ]:
 # ---------------------------------------------------------
    # PHASE 3: EXPORT TO CSV PER CONGRESS
    # ---------------------------------------------------------
    safe_filename = congress_label.split(" (")[0].replace(" ", "_").lower()
    final_filename = f"ldr_senate_bills_{safe_filename}.csv"

    df = pd.DataFrame(all_bills_data)
    df.to_csv(final_filename, index=False)
    print(f"✅ Data for {congress_label} successfully saved to {final_filename}!\n")

driver.quit()
print("All target Congresses have been successfully scraped. Your Machine Learning dataset is ready!")